In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [ ]:
donor = 'PD60966'

wd = '../data/LCM_analysis/'

In [ ]:
## Get SSM file
# ID - s...
# name - location ID
# var_reads - alt_counts
# total_reads - alt_counts + ref_counts
# var_read_prob for for CNV version; alt allele likelihood given CNA (how to estimate this?)
# pairtree paper says trisomy ~ 0.33 but could also be 0.66?
# for copy number - seems super messy to include; phasing sensitivty can get
# completely out of whack
#  IF included, ONLY use trisomy or monosomy events

pyclone_input = pd.read_csv(wd+'pyclone_vi/{}_input.tsv'.format(donor), sep = '\t')
samples = sorted(list(set(pyclone_input['sample_id'])))
pyclone_input['name'] = pyclone_input['mutation_id']

pyclone_input['total_reads'] = pyclone_input['alt_counts']+pyclone_input['ref_counts']
pyclone_input['var_reads'] = pyclone_input['alt_counts']
pyclone_input['var_read_prob'] = 0.5 # coerced because we stick to diploid regions

var_reads = pyclone_input.pivot(index = ['name'], columns = 'sample_id', values = 'var_reads')
total_reads = pyclone_input.pivot(index = ['name'], columns = 'sample_id', values = 'total_reads')
var_read_prob = pyclone_input.pivot(index = ['name'], columns = 'sample_id', values = 'var_read_prob')

# ensure indexes are the same for merging
print((var_reads.index == total_reads.index).all() & (var_read_prob.index == total_reads.index).all())

ssm = pd.concat({
    'var_reads':var_reads[samples].astype(str).agg(','.join, axis=1),
    'total_reads':total_reads[samples].astype(str).agg(','.join, axis=1),
    'var_read_prob':var_read_prob[samples].astype(str).agg(','.join, axis=1)
    }, axis=1).reset_index().rename(columns={'index':'name'})

ssm['id'] = 's'+ssm.index.astype(str)

ssm[['id', 'name', 'var_reads', 'total_reads', 'var_read_prob']]\
    .to_csv(wd+'pyclone_vi/{}_pairtree_input.tsv'.format(donor), sep = '\t')

## Get params.json
# sample - same order as SSM
# clusters - list of lists
# garbage - variants you'd like to remove

# check distribution of assignment probabilities
#sns.displot(pyclone_output[['mutation_id']].drop_duplicates()['cluster_assignment_prob'], kind = 'kde')
#plt.xlim(0.5,1)

pyclone_output = pd.read_csv(wd+'pyclone_vi/{}_output.tsv'.format(donor), sep = '\t')

# Get garbage mutation ids
garbage_mutation_ids = sorted(list(set(
    pyclone_output[pyclone_output['cluster_assignment_prob']<0.95]['mutation_id'].drop_duplicates()
    )))

# Build cluster IDs with SSM ids
cluster_ids = ssm[['name','id']].merge(
    pyclone_output[['mutation_id', 'cluster_id']]\
        .drop_duplicates().rename(columns={'mutation_id':'name'}),
        on = 'name')

# get s-ids for junk mutations
garbage = cluster_ids[cluster_ids['name'].isin(garbage_mutation_ids)]['id'].tolist()

# get s-ids for good mutations
cluster_ids = cluster_ids[~cluster_ids['id'].isin(garbage)] # remove garbage
cluster_list = sorted(list(set(cluster_ids['cluster_id'])))
clusters = []
for cluster in cluster_list:
    clusters.append(cluster_ids[cluster_ids['cluster_id']==cluster]['id'].tolist())

# make json:
params = {
    'samples':samples,
    'clusters':clusters,
    'garbage':garbage
}

# Dump to file
with open(wd+'pyclone_vi/{}_params.json'.format(donor), "w") as f:
    json.dump(params, f)

True


In [ ]:
[i for i in cluster_ids if i in garbage]

[]

In [ ]:
params = json.load(open(wd+'pyclone_vi/{}_params.json'.format(donor)))

In [ ]:
garbage = set(params['garbage'])

In [ ]:
cluster_lists = clusters["clusters"] if isinstance(clusters, dict) else clusters
clustered = [v for cl in cluster_lists for v in cl]
clustered_set = set(clustered)

In [ ]:
# Overlap with garbage
overlap = clustered_set & garbage

problems = []
if overlap:
    problems.append(f"Variants present in BOTH clusters and garbage: {sorted(overlap)}")
